## Workspace Initialization

In [2]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import sys
sys.path.append('../')

from src.eda_utils import load_and_clean_types, calculate_portfolio_metrics, get_outlier_bounds

# Replace with path to your local data file
df = load_and_clean_types('../data/insurance_data.csv')
print(f"Dataset Dimensions: {df.shape[0]} rows, {df.shape[1]} columns")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Dataset Dimensions: 10000 rows, 21 columns


## Data Summarization & Dtypes

In [5]:
# Print schema info to confirm memory types are accurate
print("--- Structural Schema Information ---")
df.info()

# Numerical Feature Descriptive Summary
print("--- Descriptive Statistics for Financial/Value Metrics ---")
numerical_features = [
    'AnnualPremium', 
    'TotalPremium', 
    'TotalClaims', 
    'ClaimAmount', 
    'CustomValueEstimate', 
    'Deductible', 
    'RiskScore', 
    'AnnualIncome'
]
print(df[numerical_features].describe())


--- Structural Schema Information ---
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   CustomerID           10000 non-null  str     
 1   Age                  10000 non-null  int64   
 2   Gender               10000 non-null  category
 3   Province             10000 non-null  category
 4   VehicleType          10000 non-null  str     
 5   AnnualIncome         10000 non-null  int64   
 6   RiskScore            10000 non-null  int64   
 7   AnnualPremium        10000 non-null  int64   
 8   Deductible           10000 non-null  int64   
 9   NCD                  10000 non-null  int64   
 10  PastClaims           10000 non-null  int64   
 11  Claimed              10000 non-null  bool    
 12  ClaimAmount          10000 non-null  float64 
 13  TotalPremium         10000 non-null  int64   
 14  TotalClaims          10000 non-null  float64

## Data Quality Assessment

In [6]:
missing_summary = df.isnull().sum()
missing_percentage = (missing_summary / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_summary,
    'Percentage (%)': missing_percentage
}).sort_values(by='Missing Count', ascending=False)

print("--- Missing Value Matrix (Top Fields) ---")
print(missing_df[missing_df['Missing Count'] > 0])

--- Missing Value Matrix (Top Fields) ---
Empty DataFrame
Columns: [Missing Count, Percentage (%)]
Index: []


## Univariate Analysis

In [7]:
print("--- Distribution Percentiles for Core Variables ---")
for col in ['TotalPremium', 'TotalClaims']:
    print(f"\n{col} Discretization:")
    print(df[col].quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

print("\n--- Top Categorical Counts (Client Demographics & Cover) ---")
for col in ['Gender', 'CoverType', 'Province']:
    print(f"\nValue Proportions for {col}:")
    print(df[col].value_counts(normalize=True).head(5))

--- Distribution Percentiles for Core Variables ---

TotalPremium Discretization:
0.10    1810.90
0.25    2028.00
0.50    2307.00
0.75    2676.00
0.90    3461.30
0.95    4348.00
0.99    4852.02
Name: TotalPremium, dtype: float64

TotalClaims Discretization:
0.10        0.00
0.25        0.00
0.50        0.00
0.75        0.00
0.90     5369.50
0.95     9200.75
0.99    18606.67
Name: TotalClaims, dtype: float64

--- Top Categorical Counts (Client Demographics & Cover) ---

Value Proportions for Gender:
Gender
Female    0.5138
Male      0.4862
Name: proportion, dtype: float64

Value Proportions for CoverType:
CoverType
Comprehensive               0.4720
Third Party Fire & Theft    0.3453
Third Party Only            0.1827
Name: proportion, dtype: float64

Value Proportions for Province:
Province
Addis Ababa    0.3567
Oromia         0.2446
Amhara         0.1999
Somali         0.1184
Tigray         0.0804
Name: proportion, dtype: float64


## Bivariate / Multivariate Analysis & Postal Metrics

In [9]:
# 1. Calculate the calculated fields (just in case they need to be re-run)
df['LossRatio'] = df['TotalClaims'] / df['TotalPremium']
df['Margin'] = df['TotalPremium'] - df['TotalClaims']

print("--- Correlation Matrix ---")
# Removed 'SumInsured' and added 'AnnualPremium', 'ClaimAmount', 'Deductible', 'RiskScore'
corr_fields = [
    'AnnualPremium', 'TotalPremium', 'TotalClaims', 
    'ClaimAmount', 'CustomValueEstimate', 'Deductible', 
    'RiskScore', 'LossRatio', 'Margin'
]
print(df[corr_fields].corr().round(3))

print("\n--- Top 10 Highest-Risk Postal Codes (Aggregated by Claim Volatility) ---")
# Using your dataset's exact 'ZipCode' column instead of 'PostalCode'
postal_risk = df.groupby('ZipCode', observed=True).agg(
    Total_Premium=('TotalPremium', 'sum'),
    Total_Claims=('TotalClaims', 'sum'),
    Average_Loss_Ratio=('LossRatio', 'mean'),
    Policy_Count=('CustomerID', 'count')  # Using CustomerID to count rows
).query('Policy_Count > 10').sort_values(by='Average_Loss_Ratio', ascending=False)

print(postal_risk.head(10))

--- Correlation Matrix ---
                     AnnualPremium  TotalPremium  TotalClaims  ClaimAmount  \
AnnualPremium                1.000         1.000        0.331        0.331   
TotalPremium                 1.000         1.000        0.331        0.331   
TotalClaims                  0.331         0.331        1.000        1.000   
ClaimAmount                  0.331         0.331        1.000        1.000   
CustomValueEstimate          0.715         0.715        0.182        0.182   
Deductible                  -0.004        -0.004       -0.040       -0.040   
RiskScore                    0.822         0.822        0.365        0.365   
LossRatio                    0.231         0.231        0.957        0.957   
Margin                      -0.151        -0.151       -0.983       -0.983   

                     CustomValueEstimate  Deductible  RiskScore  LossRatio  \
AnnualPremium                      0.715      -0.004      0.822      0.231   
TotalPremium                       0

## Geographic Trends

In [11]:
# Calculate provincial trends using your exact dataset column names
province_trends = df.groupby('Province', observed=True).agg(
    Mean_Premium=('TotalPremium', 'mean'),
    Median_Value=('CustomValueEstimate', 'median'),
    Total_Claims=('TotalClaims', 'sum'),
    Total_Premium=('TotalPremium', 'sum'),
    # Count the rows using CustomerID
    Policy_Count=('CustomerID', 'count'),
    # Find the most common car make using your exact column name: 'AutoMake'
    Top_Vehicle_Make=('AutoMake', lambda x: x.mode()[0] if not x.mode().empty else "Unknown")
)

# Calculate the true group-level Loss Ratio safely without internal indexing bugs
province_trends['Loss_Ratio'] = province_trends['Total_Claims'] / province_trends['Total_Premium']

# Reorder columns for a clean presentation layout
province_trends = province_trends[['Policy_Count', 'Mean_Premium', 'Median_Value', 'Loss_Ratio', 'Top_Vehicle_Make']]

print("--- Provincial Risk & Market Trends ---")
print(province_trends.round(3))

--- Provincial Risk & Market Trends ---
             Policy_Count  Mean_Premium  Median_Value  Loss_Ratio  \
Province                                                            
Addis Ababa          3567      2497.161       28752.0       0.522   
Amhara               1999      2465.516       28660.0       0.478   
Oromia               2446      2481.465       28126.5       0.537   
Somali               1184      2521.101       28201.5       0.612   
Tigray                804      2475.985       28740.0       0.526   

            Top_Vehicle_Make  
Province                      
Addis Ababa           Toyota  
Amhara                Toyota  
Oromia                Toyota  
Somali                Toyota  
Tigray               Hyundai  


## Outlier Detection

In [12]:
print("--- Tukey Interquartile Outlier Identification Bounds ---")
outlier_targets = ['TotalPremium', 'TotalClaims', 'CustomValueEstimate']

for target in outlier_targets:
    lower, upper = get_outlier_bounds(df[target])
    outliers = df[(df[target] < lower) | (df[target] > upper)]
    pct_outliers = (len(outliers) / len(df)) * 100
    print(f"\nFeature: {target}")
    print(f"  Valid Operational Interval: [{lower:.2f} to {upper:.2f}]")
    print(f"  Outlier Row Count: {len(outliers)} ({pct_outliers:.2f}% of total data)")

--- Tukey Interquartile Outlier Identification Bounds ---

Feature: TotalPremium
  Valid Operational Interval: [1056.00 to 3648.00]
  Outlier Row Count: 950 (9.50% of total data)

Feature: TotalClaims
  Valid Operational Interval: [0.00 to 0.00]
  Outlier Row Count: 1535 (15.35% of total data)

Feature: CustomValueEstimate
  Valid Operational Interval: [-16474.62 to 84638.38]
  Outlier Row Count: 623 (6.23% of total data)
